In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Platform.LinAlg;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Solution.LevelSetTools;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
#r "LsTest.dll"
using BoSSS.Application.LsTest;

# Load sessions

Init Workflowmanagement and Project

In [ ]:
BoSSSshell.WorkflowMgm.Init("SwirlingFlow_SpatialConvergence");
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination);
sessions

In [ ]:
using System.IO;

In [ ]:
var tab = sessions.GetSessionTable();

In [ ]:
var Evos = tab.GetColumn<string>("id:Evo").Distinct();
Evos.ForEach(e => Console.WriteLine(e));

var tRes = tab.GetColumn<double>("id:dt").Distinct();
tRes.ForEach(e => Console.WriteLine(e));

var pRes = tab.GetColumn<int>("id:Degree").Distinct();
pRes.ForEach(e => Console.WriteLine(e));

var hRes = tab.GetColumn<int>("id:Res").Distinct();
hRes.ForEach(e => Console.WriteLine(e));

In [ ]:
List<Plot2Ddata> plts = new List<Plot2Ddata>();

foreach(string Evo in Evos){
    foreach(int p in pRes){
        foreach(double dt in tRes){

            // select and sort
            var sess = sessions.Where(s => Convert.ToString(s.KeysAndQueries["id:Evo"]) == Evo &&  Convert.ToDouble(s.KeysAndQueries["id:dt"]) == dt && Convert.ToInt32(s.KeysAndQueries["id:Degree"]) == p).ToArray();
            if(sess.Count() == 0) continue;
            Array.Sort(sess, (a,b) => Convert.ToDouble(a.KeysAndQueries["id:Res"]).CompareTo(Convert.ToDouble(b.KeysAndQueries["id:Res"])));
            sess.ForEach(s => Console.WriteLine(s.Name));

            var Ref = new BoSSS.Application.LsTest.LevelSetUnitTests.LevelSetSwirlingFlowTest(2, p, false, 0.0); // to get the "Phi" Functions etc.
            var RefC = BoSSS.Application.LsTest.LevelSetUnitTests.LSTstObj2CtrlObj(Ref, int.MaxValue, LevelSetEvolution.None, BoSSS.Solution.XdgTimestepping.LevelSetHandling.None, hRes.Max(), 0);
            var grd = RefC.GridFunc(); // evaluate on the same grid

            Func<ITimestepInfo, double> errorFunctional = ts => {
                var PhiDG = ts.Fields.Single(f => f.Identification == "PhiDG");
                var PhiCG = ts.Fields.Single(f => f.Identification == "Phi");

                var PhiDGFine = new LevelSet(new Basis(grd, PhiDG.Basis.Degree), "PhiDGFine");
                var PhiCGFine = new LevelSet(new Basis(grd, PhiCG.Basis.Degree), "PhiFine");
                PhiDGFine.ProjectFromForeignGrid(1.0, (SinglePhaseField)PhiDG);
                // PhiCGFine.ProjectFromForeignGrid(1.0, (SinglePhaseField)PhiCG);
                PhiCGFine.ProjectField(1.0, Ref.GetPhi()[0].Vectorize().SetTime(0.0));

                LevelSetTracker LsTrk = new LevelSetTracker((GridData)PhiCGFine.GridDat, XQuadFactoryHelper.MomentFittingVariants.Saye, 1, new string[] { "A", "B" }, (ILevelSet)PhiCGFine);
                LsTrk.UpdateTracker(0.0);

                double err = 0.0;
                err = PhiDGFine.L2Error(Ref.GetPhi()[0].Vectorize().SetTime(0.0), 16, new BoSSS.Foundation.Quadrature.CellQuadratureScheme(true, LsTrk.Regions.GetNearFieldMask(1)));
                return err;
            };

            var errs = sess.Select(s => errorFunctional(s.Timesteps.Last())).ToArray();
            errs.ForEach(err => Console.WriteLine(err));

            var plt = sess.ToGridConvergenceData(errorFunctional);
            plts.Add(plt);

            plt.Title = "dt: " + dt + " Degree: " + p + " Evo: " + Evo;
        }
    }
}

In [ ]:
foreach(var plt in plts){
    Console.WriteLine("Slope: " + plt.Regression().First().Value);
    display(plt.PlotNow());
}

In [ ]:
List<Plot2Ddata> plts2 = new List<Plot2Ddata>();

foreach(string Evo in Evos){
    foreach(int p in pRes){
        foreach(double dt in tRes){

            // select and sort
            var sess = sessions.Where(s => Convert.ToString(s.KeysAndQueries["id:Evo"]) == Evo &&  Convert.ToDouble(s.KeysAndQueries["id:dt"]) == dt && Convert.ToInt32(s.KeysAndQueries["id:Degree"]) == p).ToArray();
            if(sess.Count() == 0) continue;
            Array.Sort(sess, (a,b) => Convert.ToDouble(a.KeysAndQueries["id:Res"]).CompareTo(Convert.ToDouble(b.KeysAndQueries["id:Res"])));
            sess.ForEach(s => Console.WriteLine(s.Name));


            var plt = sess.ToEstimatedGridConvergenceData("PhiDG");
            plts2.Add(plt);

            plt.Title = "dt: " + dt + " Degree: " + p + " Evo: " + Evo;
        }
    }
}

In [ ]:
foreach(var plt in plts2){
    Console.WriteLine("Slope: " + plt.Regression().First().Value);
    display(plt.PlotNow());
}